In [ ]:
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

digits = load_digits()
y = (digits.target == 8).astype(int)  # ¿la imagen es un 8?
X_tr, X_te, y_tr, y_te = train_test_split(
    digits.data, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
import numpy as np

print("imágenes totales:", len(digits.data))
print("imágenes en test:", len(y_te))
print("8s en test:", y_te.sum())

pred_perezoso = np.zeros_like(y_te)  # "nunca es 8"
acc_perezoso = (pred_perezoso == y_te).mean()
print(f"accuracy del perezoso: {acc_perezoso}")

imágenes totales: 1797
imágenes en test: 540
8s en test: 52
accuracy del perezoso: 0.9037037037037037


**1(a)** El test tiene **540** imágenes y **52** son un 8.

**1(b)** El perezoso acierta **0.904** (488 de 540).

**1(c)** Sale alto solo porque el 90% de las imágenes no son 8, así que decir siempre "no" ya acierta casi todo, pero no encuentra ni uno de los 52 ochos (recall 0), que es justo lo que se le pide.

In [ ]:
bosque = RandomForestClassifier(n_estimators=200,
                                random_state=0).fit(X_tr, y_tr)
prob = bosque.predict_proba(X_te)[:, 1]

def resumen(umbral):
    pred = (prob >= umbral).astype(int)
    tp = int(((pred == 1) & (y_te == 1)).sum())
    fp = int(((pred == 1) & (y_te == 0)).sum())
    fn = int(((pred == 0) & (y_te == 1)).sum())
    acc = (pred == y_te).mean()
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    print(f"umbral {umbral}: accuracy={acc:.3f}  TP={tp}  FP={fp}  FN={fn}  "
          f"precision={precision:.3f}  recall={recall:.3f}")
    return tp, fp, fn

tp5, fp5, fn5 = resumen(0.5)
print(f"de {tp5 + fp5} alertas, {tp5} eran reales")
print(f"de {tp5 + fn5} 8s reales, encontró {tp5}")

umbral 0.5: accuracy=0.961  TP=31  FP=0  FN=21  precision=1.000  recall=0.596
de 31 alertas, 31 eran reales
de 52 8s reales, encontró 31


**2(a)** Accuracy del bosque: **0.961**. Conteos: **TP = 31, FP = 0, FN = 21**.

**2(b)**
- Precision = 31 / 31 = 1.0 en "de 31 alertas, 31 eran reales".
- Recall = 31 / 52 = 0.596 en "de 52 8s reales, encontró 31".

**2(c)** No. Que la precision sea 1.0 solo dice que cuando avisa no se equivoca, pero avisa poco. El recall es **0.596**: se le escaparon **21 de los 52** ochos (FN = 21), casi la mitad.

In [ ]:
tp3, fp3, fn3 = resumen(0.3)
print(f"de {tp3 + fp3} alertas, {tp3} eran reales")
print(f"de {tp3 + fn3} 8s reales, encontró {tp3}")

umbral 0.3: accuracy=0.970  TP=42  FP=6  FN=10  precision=0.875  recall=0.808
de 48 alertas, 42 eran reales
de 52 8s reales, encontró 42


**3(a)** Con umbral 0.3: **TP = 42, FP = 6, FN = 10**, precision = 42 / 48 = **0.875** y recall = 42 / 52 = **0.808**.

**3(b)** Se ganó recall: encuentra 42 ochos en vez de 31, y los que se escapan bajan de 21 a 10. Lo que se pagó son 6 falsas alarmas que antes no había, por eso la precision baja de 1.0 a 0.875. La accuracy hasta sube un poco (0.961 → 0.970).

**3(c)** Me quedo con **0.3**. Acá lo caro es que se escape un 8 (el paquete se va a otra ciudad), y revisar una alerta falsa cuesta segundos. Con 0.5 se pierden 21 ochos; con 0.3 se pierden 10. El costo de ese cambio son 6 alertas extra para revisar a mano, y 11 paquetes bien enviados valen mucho más que unos segundos por cada una de esas 6 revisiones.

## Uso de IA e internet

Se utilizó IA únicamente para resolver dudas puntuales de sintaxis y manejo de código en Python/NumPy:

- **Conteo vectorial con máscaras booleanas (Ejercicio 2 y 3):** Se consultó la sintaxis para calcular directamente los conteos de la matriz de confusión (TP, FP, FN) de forma vectorizada sin usar bucles. Se repasó el uso del operador bitwise `&` entre arrays de NumPy (`((pred == 1) & (y_te == 1)).sum()`) y la conversión a tipo entero.

- **Sintaxis de probabilidades y umbralización manual (Ejercicio 2):** Se consultó cómo extraer únicamente la columna de probabilidades de la clase positiva en scikit-learn (`predict_proba(X_te)[:, 1]`) y cómo binarizar el vector continuo aplicando un corte personalizado con `(prob >= umbral).astype(int)`.

- **Creación de arrays base (Ejercicio 1):** Se revisó la sintaxis de NumPy para generar un vector de ceros con las mismas dimensiones y estructura que las etiquetas reales (`np.zeros_like(y_te)`) para construir la predicción del clasificador base.
